# DBSCAN for Anomaly Detection

DBSCAN (covered in detail in Section 15) is a clustering algorithm that explicitly identifies **noise points** — these are the anomalies. Unlike Isolation Forest or LOF which produce a continuous score, DBSCAN provides a hard binary label: cluster member or noise.

---

## Table of Contents
1. [DBSCAN as an Anomaly Detector](#1-idea)
2. [Why Noise Points Are Anomalies](#2-noise)
3. [Choosing ε and min_samples for Anomaly Detection](#3-params)
4. [Strengths and Weaknesses](#4-strengths)
5. [DBSCAN vs Isolation Forest vs LOF](#5-comparison)
6. [Implementation on Synthetic Data](#6-synthetic)
7. [Implementation on Real Data (Credit Card Fraud)](#7-real)
8. [Summary](#8-summary)

---
## 1. DBSCAN as an Anomaly Detector

DBSCAN clusters data based on density — regions where many points are packed within a radius ε of each other. Points that don't belong to any dense region are labeled **-1 (noise)**.

### The Core Insight for Anomaly Detection

> Normal points form dense clusters. Anomalies are isolated — they don't have enough neighbors within radius ε to be core points, and they aren't close enough to any core point to be border points.

### Three Point Types Recap

| Type | Condition | Role in Anomaly Detection |
|---|---|---|
| **Core point** | ≥ min_samples neighbors within ε | Normal — deep inside a cluster |
| **Border point** | Within ε of a core point, but < min_samples own neighbors | Normal — on the edge of a cluster |
| **Noise point** | Neither core nor border | **Anomaly** — labeled -1 |

### What DBSCAN Provides That Others Don't

1. **Arbitrary cluster shapes** — detects anomalies even when normal data is non-convex (rings, crescents)
2. **Structural context** — you get cluster membership, not just an anomaly score
3. **No training phase** — purely unsupervised, no contamination parameter needed

---
## 2. Why Noise Points Are Anomalies

### Formal Definition

A noise point $p$ satisfies:

$$|N_\varepsilon(p)| < \text{min\_samples} \quad \text{AND} \quad p \notin N_\varepsilon(q) \text{ for any core point } q$$

It has **too few neighbors** to be part of a dense region, and it's **too far** from any core point to be a border point.

### The Density Interpretation

Define the local density of point $p$ as:

$$\rho(p) = \frac{|N_\varepsilon(p)|}{V_\varepsilon}$$

where $V_\varepsilon$ is the volume of the ε-ball (constant for all points).

- Normal points: $\rho(p) \geq \text{min\_samples} / V_\varepsilon$ (high local density)
- Noise/anomaly points: $\rho(p) < \text{min\_samples} / V_\varepsilon$ (low local density)

This is fundamentally a **density threshold** decision — simpler than LOF but effective.

---
## 3. Choosing ε and min_samples for Anomaly Detection

### Effect on Anomaly Detection

**ε too small**: Many points become noise → too many false positives

**ε too large**: Everything merges into one cluster → anomalies absorbed into it → false negatives

**min_samples too small**: Even outliers become core points → anomalies missed

**min_samples too large**: Many border points become noise → too many false positives

### k-Distance Plot (optimal ε)

1. Set $k = \text{min\_samples} - 1$
2. For each point, compute distance to its $k$-th nearest neighbor
3. Sort distances ascending and plot
4. The elbow = good ε

### Rule of Thumb for min_samples

$$\text{min\_samples} \approx 2 \times d$$

where $d$ is the number of features. For anomaly detection specifically, higher min_samples means stricter density threshold — better at catching subtle anomalies.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# Normal data: two clusters with different shapes
theta = np.linspace(0, 2*np.pi, 200)
ring = np.c_[3*np.cos(theta) + np.random.randn(200)*0.2,
             3*np.sin(theta) + np.random.randn(200)*0.2]
core_cluster = np.random.randn(100, 2) * 0.4
X_normal_d = np.vstack([ring, core_cluster])

# Anomalies: scattered random points
X_anom_d = np.random.uniform(-5, 5, (15, 2))

X_all_d = np.vstack([X_normal_d, X_anom_d])
y_true_d = np.array([0]*len(X_normal_d) + [1]*len(X_anom_d))

X_scaled_d = StandardScaler().fit_transform(X_all_d)

# k-distance plot
min_s = 5
k = min_s - 1
nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled_d)
dists, _ = nbrs.kneighbors(X_scaled_d)
k_dists = np.sort(dists[:, -1])

d2 = np.diff(np.diff(k_dists))
elbow = np.argmax(d2) + 1
eps_auto = k_dists[elbow]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(k_dists, color='steelblue')
axes[0].axhline(eps_auto, color='red', linestyle='--', label=f'ε = {eps_auto:.3f}')
axes[0].set_xlabel('Points sorted by k-distance')
axes[0].set_ylabel(f'{k}-NN distance')
axes[0].set_title('k-Distance Plot for ε Selection')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Apply DBSCAN
db = DBSCAN(eps=eps_auto, min_samples=min_s)
labels_d = db.fit_predict(X_scaled_d)
noise_mask = labels_d == -1
n_noise = noise_mask.sum()
n_clusters = len(set(labels_d)) - 1

axes[1].scatter(X_scaled_d[~noise_mask, 0], X_scaled_d[~noise_mask, 1],
                c=labels_d[~noise_mask], cmap='tab10', s=20, alpha=0.7, label='Cluster member')
axes[1].scatter(X_scaled_d[noise_mask, 0], X_scaled_d[noise_mask, 1],
                c='black', s=100, marker='x', lw=2, zorder=5, label=f'Noise/Anomaly ({n_noise})')
axes[1].scatter(X_scaled_d[y_true_d==1, 0], X_scaled_d[y_true_d==1, 1],
                c='none', edgecolors='red', s=200, lw=2, label='True anomaly')
axes[1].set_title(f'DBSCAN Anomaly Detection\n{n_clusters} clusters, {n_noise} noise points')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dbscan_anomaly_detection.png', dpi=120, bbox_inches='tight')
plt.show()

from sklearn.metrics import precision_score, recall_score
y_pred_d = noise_mask.astype(int)
print(f"DBSCAN — Precision: {precision_score(y_true_d, y_pred_d):.3f}, Recall: {recall_score(y_true_d, y_pred_d):.3f}")
print(f"True anomalies caught: {(noise_mask & (y_true_d==1)).sum()} of {(y_true_d==1).sum()}")

---
## 4. Strengths and Weaknesses

### Strengths

1. **No score threshold to tune** — noise is determined by ε and min_samples directly
2. **Works with any cluster shape** — normal data can be rings, crescents, spirals
3. **No contamination rate needed** — unlike Isolation Forest, doesn't require knowing the anomaly proportion
4. **Interpretable** — flagged points are explicitly in no dense region
5. **Deterministic** — same result every run

### Weaknesses

1. **ε and min_samples are sensitive** — small changes in ε can dramatically change results
2. **Single global density threshold** — fails when normal data has varying densities (some clusters dense, others sparse)
3. **Doesn't produce a continuous score** — can't rank anomalies by severity (just in/out)
4. **High-dimensional data** — ε-neighborhoods become meaningless in high dimensions
5. **Can't score new points** — DBSCAN requires re-fitting on the full dataset; no `predict()` for new samples

### The Varying Density Problem

If normal data has regions of very different density (a tight cluster and a loose cluster), a single ε:
- Small enough for the tight cluster → many points in the loose cluster become noise (false positives)
- Large enough for the loose cluster → anomalies near the tight cluster get absorbed (false negatives)

**Solution**: Use **HDBSCAN** (Hierarchical DBSCAN) which adapts to varying densities.

---
## 5. DBSCAN vs Isolation Forest vs LOF

| Aspect | DBSCAN | Isolation Forest | LOF |
|---|---|---|---|
| Output | Hard labels (in/out) | Continuous score | Continuous score |
| Cluster-aware | Yes — knows the clusters | No | No |
| Arbitrary shapes | **Yes** | No (global) | Partial |
| Varying density | **No** | No | **Yes** |
| Needs contamination? | No | Yes | Yes |
| Predict new points | No (re-fit) | Yes | Yes (novelty mode) |
| High-dim data | **No** | **Yes** | No |
| Speed | $O(n \log n)$ | $O(n \log n)$ | $O(n^2)$ |

**Use DBSCAN for anomaly detection when:**
- Normal data has clear spatial clusters
- Data is 2D–10D (distance metrics work well)
- You want to understand cluster structure, not just flag outliers
- Anomalies are spatially isolated from normal clusters

In [ ]:
# Side-by-side comparison: DBSCAN vs Isolation Forest on ring+core data
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
eps_vals = [0.15, eps_auto, 0.5]
labels_eps = []

for ax, eps_v in zip(axes, eps_vals):
    db_v = DBSCAN(eps=eps_v, min_samples=5)
    lbl_v = db_v.fit_predict(X_scaled_d)
    noise_v = lbl_v == -1
    n_c = len(set(lbl_v)) - 1
    labels_eps.append(noise_v)

    ax.scatter(X_scaled_d[~noise_v, 0], X_scaled_d[~noise_v, 1],
               c=lbl_v[~noise_v], cmap='tab10', s=15, alpha=0.6)
    ax.scatter(X_scaled_d[noise_v, 0], X_scaled_d[noise_v, 1],
               c='black', s=80, marker='x', lw=2, label=f'Noise ({noise_v.sum()})')
    ax.scatter(X_scaled_d[y_true_d==1, 0], X_scaled_d[y_true_d==1, 1],
               c='none', edgecolors='red', s=200, lw=2)
    prec = precision_score(y_true_d, noise_v.astype(int), zero_division=0)
    rec = recall_score(y_true_d, noise_v.astype(int))
    ax.set_title(f'ε={eps_v:.2f} → {n_c} clusters\nP={prec:.2f}, R={rec:.2f}', fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('DBSCAN ε Sensitivity (red circles = true anomalies)', fontsize=12)
plt.tight_layout()
plt.savefig('dbscan_epsilon_compare.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Implementation on Synthetic Data

In [ ]:
# Multi-algorithm comparison on various data shapes
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)

def add_outliers(X, n_out=20):
    lims = X.max(axis=0) - X.min(axis=0)
    outliers = np.random.uniform(X.min(axis=0) - lims*0.5,
                                  X.max(axis=0) + lims*0.5, (n_out, X.shape[1]))
    return np.vstack([X, outliers]), np.array([0]*len(X) + [1]*n_out)

datasets = [
    ('Blobs', *add_outliers(make_blobs(300, centers=3, cluster_std=0.5, random_state=42)[0])),
    ('Moons', *add_outliers(make_moons(300, noise=0.05, random_state=42)[0])),
    ('Circles', *add_outliers(make_circles(300, noise=0.03, factor=0.5, random_state=42)[0])),
]

fig, axes = plt.subplots(3, 2, figsize=(12, 15))

for row, (name, X_d, y_d) in enumerate(datasets):
    X_s = StandardScaler().fit_transform(X_d)

    # DBSCAN
    nbrs2 = NearestNeighbors(n_neighbors=4).fit(X_s)
    dists2, _ = nbrs2.kneighbors(X_s)
    eps2 = np.percentile(dists2[:, -1], 90)
    db2 = DBSCAN(eps=eps2, min_samples=5)
    lbl2 = db2.fit_predict(X_s)
    noise2 = (lbl2 == -1)

    # Isolation Forest
    iso2 = IsolationForest(contamination=20/len(X_d), random_state=42)
    iso2.fit(X_s)
    lbl_iso = iso2.predict(X_s)
    noise_iso = lbl_iso == -1

    for col, (noise, title) in enumerate([(noise2, f'DBSCAN (ε={eps2:.2f})'),
                                           (noise_iso, 'Isolation Forest')]):
        ax = axes[row, col]
        ax.scatter(X_s[~noise, 0], X_s[~noise, 1], c='steelblue', s=12, alpha=0.5, label='Normal')
        ax.scatter(X_s[noise, 0], X_s[noise, 1], c='red', s=60, marker='x', lw=2,
                   label=f'Flagged ({noise.sum()})')
        ax.scatter(X_s[y_d==1, 0], X_s[y_d==1, 1], c='none', edgecolors='green', s=150, lw=2)
        p = precision_score(y_d, noise.astype(int), zero_division=0)
        r = recall_score(y_d, noise.astype(int))
        ax.set_title(f'{name} — {title}\nP={p:.2f}, R={r:.2f}', fontsize=10)
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('DBSCAN vs Isolation Forest (green circles = true anomalies)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('dbscan_vs_isoforest.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Implementation on Real Data

In [ ]:
# Using DBSCAN for anomaly detection on a real tabular dataset
from sklearn.datasets import load_wine
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt

wine = load_wine()
X_wine = StandardScaler().fit_transform(wine.data)

# Find ε via k-distance plot
min_s_w = 5
nbrs_w = NearestNeighbors(n_neighbors=min_s_w-1).fit(X_wine)
dists_w, _ = nbrs_w.kneighbors(X_wine)
k_dists_w = np.sort(dists_w[:, -1])
eps_w = np.percentile(k_dists_w, 85)  # 85th percentile as a reasonable ε

db_wine = DBSCAN(eps=eps_w, min_samples=min_s_w)
labels_wine = db_wine.fit_predict(X_wine)
noise_wine = labels_wine == -1

iso_wine = IsolationForest(n_estimators=200, contamination=noise_wine.mean(), random_state=42)
iso_wine.fit(X_wine)
noise_iso_wine = iso_wine.predict(X_wine) == -1

X_2d = PCA(n_components=2).fit_transform(X_wine)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (noise, title) in zip(axes, [
    (noise_wine, f'DBSCAN (ε={eps_w:.2f}, {noise_wine.sum()} anomalies)'),
    (noise_iso_wine, f'Isolation Forest ({noise_iso_wine.sum()} anomalies)')
]):
    ax.scatter(X_2d[~noise, 0], X_2d[~noise, 1],
               c=wine.target[~noise], cmap='tab10', s=25, alpha=0.7, label='Normal')
    ax.scatter(X_2d[noise, 0], X_2d[noise, 1],
               c='black', s=100, marker='x', lw=2, zorder=5, label='Anomaly')
    ax.set_title(title, fontsize=11)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Wine Dataset — DBSCAN vs Isolation Forest (PCA 2D view)', fontsize=12)
plt.tight_layout()
plt.savefig('dbscan_anomaly_wine.png', dpi=120, bbox_inches='tight')
plt.show()

overlap = (noise_wine & noise_iso_wine).sum()
print(f"Points flagged by DBSCAN:           {noise_wine.sum()}")
print(f"Points flagged by Isolation Forest: {noise_iso_wine.sum()}")
print(f"Points flagged by both:             {overlap}")
print(f"Agreement rate:                     {overlap/max(noise_wine.sum(), noise_iso_wine.sum()):.1%}")

---
## 8. Summary

### Key Points

| Concept | Detail |
|---|---|
| Anomaly = noise point | Label -1 in DBSCAN output |
| ε selection | k-distance plot elbow (k = min_samples - 1) |
| min_samples | 2 × n_features; higher = stricter density threshold |
| Strength | Arbitrary cluster shapes, no contamination parameter |
| Weakness | No continuous score, sensitive to ε, fails in high-dim |
| Varying density | Use HDBSCAN instead |

### sklearn Quick Reference
```python
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import numpy as np

X_scaled = StandardScaler().fit_transform(X)

# Choose ε via k-distance plot
k = 4  # min_samples - 1
nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled)
dists, _ = nbrs.kneighbors(X_scaled)
eps = np.percentile(np.sort(dists[:, -1]), 90)  # or use elbow

db = DBSCAN(eps=eps, min_samples=5)
labels = db.fit_predict(X_scaled)

anomalies = X_scaled[labels == -1]  # noise points = anomalies
n_anomalies = (labels == -1).sum()
```